In [ ]:
# --------------------------------------------------
# 1️⃣ Mount Google Drive (optional, for cache)
# --------------------------------------------------
from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("📦 Google Drive already mounted")

# --------------------------------------------------
# 2️⃣ Clone fenicsx-colab repository (idempotent)
# --------------------------------------------------
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

def run(cmd):
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    print("📥 Cloning fenicsx-colab...")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError("Directory exists but is not a git repository")
else:
    print("📦 Repository already exists — skipping clone")

# --------------------------------------------------
# 3️⃣ Run setup_fenicsx.py IN THIS KERNEL (CRITICAL)
# --------------------------------------------------
print("🚀 Running setup_fenicsx.py in current kernel")

# ⚙️ Configuration
USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = True    # <--- Set True to remove existing environment

# Build options
opts = []
if USE_COMPLEX:
    opts.append("--complex")
if USE_CLEAN:
    opts.append("--clean")

opts_str = " ".join(opts) if opts else ""

get_ipython().run_line_magic(
    "run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}"
)

# --------------------------------------------------
# 4️⃣ Sanity check
# --------------------------------------------------
try:
    get_ipython().run_cell_magic('fenicsx', '--info -np 4', '')
except Exception as e:
    print("⚠️ %%fenicsx magic not found:", e)

In [ ]:
%%fenicsx -np 4

import numpy as np
import time
from pathlib import Path
from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
from dolfinx import mesh, fem, io, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import ufl
import basix

COMM = MPI.COMM_WORLD
RANK = COMM.rank

def rank_print(msg):
    if RANK == 0:
        print(msg, flush=True)

# ============================================================
# 1. 메시 생성 (Disk)
# ============================================================
rank_print("="*55 + "\n  STEP 1: Mesh Generation (Disk)\n" + "="*55)
import gmsh
gmsh.initialize()
if RANK == 0:
    membrane = gmsh.model.occ.addDisk(0, 0, 0, 1, 1)
    gmsh.model.occ.synchronize()
    gmsh.model.addPhysicalGroup(2, [membrane], 1)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", 0.05)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 0.05)
    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.setOrder(1)

msh_data = io.gmsh.model_to_mesh(gmsh.model, COMM, 0, gdim=2)
domain = msh_data.mesh
ft = msh_data.facet_tags
gmsh.finalize()

Path("results").mkdir(exist_ok=True)

def phi_set(x):
    r = np.sqrt(x[0]**2 + x[1]**2)
    r0, beta = 0.5, 0.9
    b = r0 * beta
    t = np.sqrt(r0**2 - b**2)
    B = t + b**2 / t
    C = -b / t
    cond_true = B + r * C
    cond_false = np.sqrt(np.maximum(r0**2 - r**2, 0.0))
    true_indices = np.flatnonzero(r > b)
    cond_false[true_indices] = cond_true[true_indices]
    return cond_false

# ============================================================
# 2. SNES (vinewtonssls) - 제약조건이 있는 장애물 벤치마크
# ============================================================
rank_print("\n" + "="*55 + "\n  STEP 2: SNES (vinewtonssls)\n" + "="*55)

V_s = fem.functionspace(domain, ("Lagrange", 1))
fdim = domain.topology.dim - 1
domain.topology.create_connectivity(fdim, domain.topology.dim)

bdry_facets = mesh.exterior_facet_indices(domain.topology)
bdry_dofs = fem.locate_dofs_topological(V_s, fdim, bdry_facets)
u_bc_s = fem.Function(V_s)
u_bc_s.x.array[:] = 0.0
bcs_s = [fem.dirichletbc(u_bc_s, bdry_dofs)]

phi_s = fem.Function(V_s, name="obstacle")
phi_s.interpolate(phi_set)

u_max_s = fem.Function(V_s)
u_max_s.x.array[:] = PETSc.INFINITY

uh_s = fem.Function(V_s, name="SNES")
v_s = ufl.TestFunction(V_s)

F_s = ufl.inner(ufl.grad(uh_s), ufl.grad(v_s)) * ufl.dx
J_s = ufl.derivative(F_s, uh_s)

# ✅ V0.7.x 완벽 호환: NonlinearProblem이 내부적으로 안전하게 SNES 통신을 관리함
snes_options = {
    "snes_type": "vinewtonssls",
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_max_it": 200,
}

problem_snes = NonlinearProblem(
    F_s, uh_s, bcs=bcs_s, J=J_s, 
    petsc_options=snes_options, petsc_options_prefix="snes_"
)

# 내부 solver(SNES)에 직접 접근하여 하한(장애물) 및 상한 제약조건 설정
problem_snes.solver.setVariableBounds(phi_s.x.petsc_vec, u_max_s.x.petsc_vec)

problem_snes.solve()
rank_print(f"✅ SNES Converged in {problem_snes.solver.getIterationNumber()} iterations.")

with io.XDMFFile(COMM, "results/obstacle_snes.xdmf", "w") as xf:
    xf.write_mesh(domain)
    xf.write_function(uh_s)

# ============================================================
# 3. LVPP (Proximal Galerkin)
# ============================================================
rank_print("\n" + "="*55 + "\n  STEP 3: LVPP (Proximal Galerkin)\n" + "="*55)

poly_order = 1
P = basix.ufl.element("Lagrange", domain.basix_cell(), poly_order)
V_m = fem.functionspace(domain, basix.ufl.mixed_element([P, P]))

V0, _ = V_m.sub(0).collapse()
dofs_m = fem.locate_dofs_topological((V_m.sub(0), V0), fdim, bdry_facets)
u_bc_m = fem.Function(V0)
u_bc_m.x.array[:] = 0.0
bcs_m = [fem.dirichletbc(u_bc_m, dofs_m, V_m.sub(0))]

sol = fem.Function(V_m)
sol_k = fem.Function(V_m)
sol.x.array[:] = 0.0
sol_k.x.array[:] = 0.0

u, psi = ufl.split(sol)
u_k, psi_k = ufl.split(sol_k)

q_deg = 6
try:
    Qe = basix.ufl.quadrature_element(domain.topology.cell_name(), value_shape=(), degree=q_deg, scheme="default")
except:
    Qe = basix.ufl.quadrature_element(domain.topology.cell_name(), degree=q_deg)
    
Vq = fem.functionspace(domain, Qe)
phi_q = fem.Function(Vq)
phi_q.interpolate(phi_set)

dxq = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": q_deg})
v_m, w_m = ufl.TestFunctions(V_m)

alpha = fem.Constant(domain, default_scalar_type(1.0))

# Proximal Galerkin 정식화
F_m = (
    alpha * ufl.inner(ufl.grad(u), ufl.grad(v_m)) * dxq
    + psi * v_m * dxq
    + u * w_m * dxq
    - ufl.exp(psi) * w_m * dxq
    - phi_q * w_m * dxq
    - psi_k * v_m * dxq
)
J_m = ufl.derivative(F_m, sol)

lvpp_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_linesearch_type": "none",
    "snes_rtol": 1e-8,
    "snes_atol": 1e-8,
    "snes_max_it": 50,
}

problem_lvpp = NonlinearProblem(
    F_m, u=sol, bcs=bcs_m, J=J_m, 
    petsc_options=lvpp_options, petsc_options_prefix="lvpp_"
)

H1inc_form = fem.form(ufl.inner(ufl.grad(u - u_k), ufl.grad(u - u_k)) * dxq + (u - u_k)**2 * dxq)

alpha_k = 1.0
C, r_p, q_p = 1.0, 1.5, 1.5
total_newton = 0

t0 = time.time()
for k_iter in range(100):
    try:
        new_val = max(C * r_p**(q_p**k_iter) - alpha_k, C)
    except OverflowError:
        new_val = 1e5
    alpha.value = min(new_val, 1e5)
    alpha_k = float(alpha.value)

    problem_lvpp.solve()
    n_it = problem_lvpp.solver.getIterationNumber()
    total_newton += n_it

    incr = np.sqrt(COMM.allreduce(fem.assemble_scalar(H1inc_form), op=MPI.SUM))
    rank_print(f"  [LVPP] Outer {k_iter+1:2d} | α={alpha.value:.1e} | Newton: {n_it} | Incr: {incr:.3e}")

    if incr < 1e-4:
        rank_print(f"✅ LVPP Converged! Total Newton steps: {total_newton}")
        break

    sol_k.x.array[:] = sol.x.array[:]

u_out = fem.Function(V_s, name="u_lvpp")
# 튜플 인덱싱([0]) 없이 그냥 서브 함수를 바로 보간(interpolate)
u_out.interpolate(sol.sub(0))

with io.XDMFFile(COMM, "results/obstacle_lvpp.xdmf", "w") as xf:
    xf.write_mesh(domain)
    xf.write_function(u_out)

rank_print("\n" + "="*55 + f"\n  All Finished! Total time: {(time.time()-t0):.1f}s\n" + "="*55)